# Baseline CNN, trained on Kaggle

ITRI626 group project. This notebook trains the from-scratch baseline CNN on the shared
70/15/15 split of Stanford Dogs. The logic lives in `Lindani_Model`, not in these cells,
so there is one version of the code rather than one per notebook.

## Before running anything

1. **GPU on.** Right-hand panel, Session options, Accelerator, choose **GPU T4 x2** or
   **GPU P100**. Without it, one epoch takes over an hour instead of a minute or two.
   If the option is greyed out, verify your phone number at kaggle.com/settings.
2. **Add the photographs.** Top right, **Add Input**, search `stanford-dogs-dataset` and
   add *Stanford Dogs Dataset* by Jessica Li.
3. **Add the code.** Add the private dataset holding the contents of `Lindani_Model`
   (see step 2 below). If instead you have working access to GitHub, turn **Internet** on
   in Session options and the notebook will clone the code by itself.

Kaggle allows about 30 GPU hours a week and shuts down idle sessions, so for the long run
use **Save Version, Run All** and let it finish unattended.

## The rule this notebook keeps

The test set is never read during training. It is scored once, afterwards, by
`evaluate.py`. Every reproducibility detail the specification asks for is written to
`run_config.json` automatically rather than reconstructed later.

## Step 1: check the machine

Confirms a GPU is actually attached. If `cuda available` prints `False`, stop and turn the
accelerator on, otherwise everything below runs on the CPU and will be unusably slow.

In [ ]:
import torch

print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            ", torch.cuda.get_device_name(0))
    print("memory gb      ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

!ls /kaggle/input

## Step 2: get the code into the session

Two routes, and the cell below picks whichever is available without being told.

**Uploaded code, the route in use.** GitHub is blocked on the development network, so the
code is uploaded to Kaggle as a small private dataset (`train.py`, `model.py`,
`dataset.py`, `split_check.py`, `split_manifest.json` and the leaked photograph list,
about 0.15 MB zipped). Add it to this notebook as an input alongside the dogs dataset.

**Cloning from the repository.** Used automatically if no uploaded code is found. It
takes only the `Lindani_Model` folder through a sparse checkout, rather than the 800 MB
of photographs the repository also holds.

Either way the files are copied into `/kaggle/working/code/Lindani_Model`, because
`/kaggle/input` is **read-only** and the training script writes its results beside the
code.

One caution about the uploaded route: a Kaggle dataset is a snapshot. If any of these
files change on the laptop, the dataset has to be uploaded again, or this notebook will
quietly train yesterday's code.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/JP-NWU-honours-2026/Dog_Classifier.git"
BRANCH = "Lindani-branch"
WORK = "/kaggle/working/code"          # stands in for the repository root
CODE = Path(WORK) / "Lindani_Model"
NEEDED = ["train.py", "model.py", "dataset.py", "split_check.py", "split_manifest.json"]

# Kaggle nests attached inputs differently depending on how they were added, so
# look several folders deep rather than assuming one layout.
DEPTHS = ["*", "*/*", "*/*/*", "*/*/*/*"]


def find_uploaded_code():
    """Looks for the code in any attached dataset, up to four folders deep."""
    root = Path("/kaggle/input")
    for pattern in DEPTHS:
        for folder in sorted(root.glob(pattern)):
            if folder.is_dir() and all((folder / f).is_file() for f in NEEDED):
                return folder
    return None


# Start clean, so a previous half-finished attempt cannot leave stale files behind.
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)

uploaded = find_uploaded_code()

if uploaded:
    print("using uploaded code from", uploaded)
    shutil.copytree(uploaded, CODE, dirs_exist_ok=True)
else:
    print("no uploaded code found, cloning the repository instead")
    subprocess.run(["rm", "-rf", "/kaggle/working/repo"], check=False)
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
                    "--branch", BRANCH, REPO, "/kaggle/working/repo"], check=True)
    subprocess.run("cd /kaggle/working/repo && git sparse-checkout init --cone "
                   "&& git sparse-checkout set Lindani_Model && git checkout",
                   shell=True, check=True)
    shutil.copytree("/kaggle/working/repo/Lindani_Model", CODE, dirs_exist_ok=True)

print("\ncode now in", CODE)
for path in sorted(CODE.iterdir()):
    print("  ", path.name)

missing = [f for f in NEEDED if not (CODE / f).is_file()]
if missing:
    raise AssertionError(
        f"missing {', '.join(missing)}.\n"
        "If this cell cloned the repository, the branch on GitHub does not have these "
        "files yet, so push them from the laptop and run this cell again. Otherwise add "
        "the code dataset through Add Input."
    )
print("\nAll five required files are present.")

## Step 3: find the photographs

`dataset.py` looks for the breed folders in several places and takes the first one it
finds, with the `DOG_IMAGES_DIR` environment variable checked first. The cell below
searches every dataset attached to this notebook for a folder containing at least 100
folders named like `n02085620-Chihuahua`, then sets that variable.

So it does not matter which mirror of Stanford Dogs was added, or how deeply the folders
are nested inside it. It also prints every input attached, which is useful when something
is missing.

If it fails, the message says what to do: add the *Stanford Dogs Dataset* by Jessica Li
through **Add Input**.

In [ ]:
import os
from pathlib import Path

MAX_DEPTH = 7


def find_breed_folders():
    """
    Walks the attached inputs looking for the folder that holds the 120 breed
    folders, which are named like n02085620-Chihuahua.

    Kaggle mounts inputs as /kaggle/input/datasets/<owner>/<dataset>/..., so the
    folder sits five levels down rather than one or two. Walking the tree is
    safer than guessing any particular layout, and it stops at the parent of the
    breed folders rather than descending into 20,580 photographs.
    """
    root = Path("/kaggle/input")
    stack = [(root, 0)]
    while stack:
        folder, depth = stack.pop()
        try:
            children = [p for p in folder.iterdir() if p.is_dir()]
        except (PermissionError, OSError):
            continue
        breeds = [p for p in children if p.name.startswith("n0")]
        if len(breeds) >= 100:
            return folder, len(breeds)
        if depth < MAX_DEPTH:
            stack.extend((child, depth + 1) for child in children)
    return None, 0


print("Inputs attached to this notebook:")
for owner in sorted(Path("/kaggle/input").glob("*/*")):
    for dataset in sorted(owner.glob("*")):
        print("  ", dataset.relative_to("/kaggle/input"))

images_dir, breed_count = find_breed_folders()
if not images_dir:
    raise AssertionError(
        "No folder of breed folders found in any attached input. Use Add Input to add "
        "the Stanford Dogs Dataset by Jessica Li."
    )

os.environ["DOG_IMAGES_DIR"] = str(images_dir)
print("\nusing          ", images_dir)
print("breed folders  ", breed_count)

## Step 4: check the split and the photographs agree

Runs the self-test in `dataset.py`. It must report 14,355 training, 3,025 validation and
3,200 test photographs, 120 breeds, and pass its label check. If it cannot find the
photographs it will say so plainly.

In [ ]:
!cd {WORK} && python Lindani_Model/dataset.py

## Step 5: smoke test, two epochs over a handful of batches

Proves the whole path works on the GPU before committing an hour to it. Takes a minute or
two.

**Check three things in the output:**

- `Training on cuda` followed by the GPU name, not `cpu`
- `1,631,128 trainable parameters`
- the loss falls between the two epochs

Accuracy here is meaningless. Eight batches teaches the model almost nothing.

In [ ]:
!cd {WORK} && python Lindani_Model/train.py --quick --name kaggle_smoke --num-workers 2 --overwrite

## Step 6: iteration 2, two runs

Iteration 1 is finished and recorded: 50.51 per cent validation accuracy, macro F1 0.497,
top five accuracy 81.36 per cent. Its evaluation showed training loss falling to 1.16
while validation loss flattened at 1.90 from about epoch 45, so **the binding constraint
is overfitting rather than capacity**, and more epochs would achieve nothing.

The split, the input size, the augmentation and the seed stay exactly as they are, because
they are shared with JP and Sulaiman and the three way comparison depends on them. What is
ours to change is the architecture and the regularisation, so iteration 2 tests one of
each, as two separate runs, each judged on validation only.

**Run A, stronger regularisation.** Dropout 0.3 to 0.5 and weight decay 1e-4 to 1e-3, same
architecture. This attacks the overfitting the curves actually show.

**Run B, more capacity for fine detail.** A second convolution in blocks 4 and 5, taking
the model from 1.63 to 4.58 million parameters. This attacks the other finding, that top
five accuracy is 81 per cent while top one is 50, meaning the model recognises dogs but
cannot make the final distinction. With unchanged augmentation it may well overfit harder,
and that result is worth reporting either way.

Each run takes roughly 65 minutes, so expect about two and a quarter hours in total.

**The test set is still not touched.** Whichever model wins on validation gets scored on
test once, at the end, by `evaluate.py --final`.

In [ ]:
RUNS = ["iter2_regularised", "iter2_deeper"]

# Run A: stronger regularisation, same architecture as iteration 1.
!cd {WORK} && python Lindani_Model/train.py --epochs 60 --batch-size 64 --num-workers 2 \
    --name iter2_regularised --dropout 0.5 --weight-decay 1e-3 --overwrite

# Run B: a second convolution in blocks 4 and 5, 4.58 million parameters.
!cd {WORK} && python Lindani_Model/train.py --epochs 60 --batch-size 64 --num-workers 2 \
    --name iter2_deeper --convs-per-block 1,1,1,2,2 --overwrite

## Step 7: look at the run before leaving

Prints the last few epochs and the headline figures. Two things to note for the report:
whether **early stopping** triggered or the **epoch cap** was reached, and how far the
training and validation curves diverged, which is how overfitting shows up.

In [ ]:
import json

for name in RUNS:
    run = Path(WORK) / "Lindani_Model/results/runs" / name
    if not (run / "run_config.json").is_file():
        print(f"{name}: no results, the run did not finish\n")
        continue

    config = json.loads((run / "run_config.json").read_text())
    print(f"=== {name} ===")
    for key in ["trainable_parameters", "dropout", "weight_decay", "convs_per_block",
                "epochs_planned", "epochs_run", "stopped_early", "epoch_cap_reached",
                "best_epoch", "best_val_macro_f1", "total_minutes"]:
        print(f"  {key:<22} {config.get(key)}")

    rows = (run / "metrics.csv").read_text().strip().split("\n")
    print("  last three epochs:")
    for line in rows[-3:]:
        print("   ", line)
    print()

print("Iteration 1 for comparison: best_val_macro_f1 0.49688, validation accuracy 50.51 per cent")

## Step 8: take the results off Kaggle

Kaggle wipes the session when it ends. Only `/kaggle/working` survives, and only if the
notebook version is saved, so copy the run folder somewhere obvious and download it.

**What to do with each file once it is on your laptop**, in
`Lindani_Model/results/runs/baseline_cnn/`:

| File | What to do |
|---|---|
| `run_config.json` | Commit. This is the reproducibility record |
| `metrics.csv` | Commit. The training and validation curves come from this |
| `val_confusion_best.csv` | Commit. Small, and useful for error analysis |
| `best_model.pth` | **Do not commit.** Weights are ignored by git on purpose. Keep it for `evaluate.py` |

In [ ]:
import shutil

destination = Path("/kaggle/working/iteration2_runs")
shutil.rmtree(destination, ignore_errors=True)
destination.mkdir(parents=True)

for name in RUNS:
    run = Path(WORK) / "Lindani_Model/results/runs" / name
    if run.is_dir():
        shutil.copytree(run, destination / name)

for folder in sorted(destination.iterdir()):
    print(folder.name)
    for path in sorted(folder.iterdir()):
        print(f"   {path.name:<26} {path.stat().st_size / 1024:>9.0f} kB")

print("\nDownload the iteration2_runs folder from the Output tab on the right.")
print("Put each run in Lindani_Model/results/runs/<name>/ on the laptop, keeping the")
print("four files only. Commit everything except best_model.pth.")